# Solution 21: Multi-Epoch Ensemble + Hybrid TTA (Best)

I loaded the epoch 3, 4, 5, and 6 adapters from Solution 19, ran hybrid TTA with each, and averaged the digit logits across all epochs before taking the argmax. The ensemble smooths out per-epoch variance.

Per-epoch validation accuracy: E3=0.8931, E4=0.8998, E5=0.9094, E6=0.9094.

This gave the best score of the project: 0.92555.

**Score: 0.92555**

In [1]:
!pip install -q "transformers==4.47.0"
!pip uninstall -y torchao 2>/dev/null
!pip install -q accelerate peft bitsandbytes pillow tqdm pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 150.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 126.9 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os, json, random, math
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from itertools import permutations

import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
DATA_DIR = "/content/drive/MyDrive/pixels-to-predictions"
IMG_BASE = os.path.join(DATA_DIR, "images")
CKPT_DIR = "/content/drive/MyDrive/pixels-to-predictions/checkpoints_v19"

ENSEMBLE_EPOCHS = [3, 4, 5, 6]
IMAGE_LONGEST_EDGE = 512
MAX_SEQ_LEN = 1024
TTA_BATCH_SIZE = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():

for e in ENSEMBLE_EPOCHS:
    p = os.path.join(CKPT_DIR, f"epoch_{e}", "adapter_config.json")
    assert os.path.exists(p), f"Missing checkpoint: epoch_{e}"
    print(f"  epoch_{e}: OK")
print(f"\nEnsembling {len(ENSEMBLE_EPOCHS)} epochs: {ENSEMBLE_EPOCHS}")

Device: cuda
  epoch_3: OK
  epoch_4: OK
  epoch_5: OK
  epoch_6: OK

Ensembling 4 epochs: [3, 4, 5, 6]


In [4]:
val_df = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print(f"Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Val num_choices: {dict(val_df['num_choices'].value_counts().sort_index())}")
print(f"Test num_choices: {dict(test_df['num_choices'].value_counts().sort_index())}")

Val: 1048 | Test: 1008
Val num_choices: {2: np.int64(244), 3: np.int64(508), 4: np.int64(252), 5: np.int64(44)}
Test num_choices: {2: np.int64(272), 3: np.int64(438), 4: np.int64(260), 5: np.int64(38)}


In [5]:
def build_prompt(row):
    """Same prompt as v16/v19."""
    choices = json.loads(row['choices'])
    choices_text = "\n".join(f"({i}) {c}" for i, c in enumerate(choices))
    parts = []
    if pd.notna(row.get('hint', None)) and str(row['hint']).strip():
        parts.append(f"Context: {row['hint'].strip()}")
    if pd.notna(row.get('lecture', None)) and str(row['lecture']).strip():
        parts.append(f"Background: {row['lecture'].strip()}")
    parts.append(f"Question: {row['question'].strip()}")
    parts.append(f"Choices:\n{choices_text}")
    parts.append("Answer with ONLY the number of the correct choice.")
    return "\n\n".join(parts)

## Load Base Model + All Epoch Adapters

In [6]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.size = {"longest_edge": IMAGE_LONGEST_EDGE}

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto")

first_epoch = ENSEMBLE_EPOCHS[0]
model = PeftModel.from_pretrained(
    base_model,
    os.path.join(CKPT_DIR, f"epoch_{first_epoch}"),
    adapter_name=f"epoch_{first_epoch}",
)
print(f"Loaded epoch_{first_epoch} as base adapter")

for e in ENSEMBLE_EPOCHS[1:]:
    model.load_adapter(
        os.path.join(CKPT_DIR, f"epoch_{e}"),
        adapter_name=f"epoch_{e}",
    )
    print(f"Loaded epoch_{e}")

model.eval()
print(f"\nAll {len(ENSEMBLE_EPOCHS)} adapters loaded. Ready for ensemble.")

DIGIT_TOKEN_IDS = [processor.tokenizer.encode(str(d), add_special_tokens=False)[0] for d in range(5)]
print(f"Digit token IDs: {dict(enumerate(DIGIT_TOKEN_IDS))}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Loaded epoch_3 as base adapter
Loaded epoch_4
Loaded epoch_5
Loaded epoch_6

All 4 adapters loaded. Ready for ensemble.
Digit token IDs: {0: 32, 1: 33, 2: 34, 3: 35, 4: 36}


## Hybrid TTA - Returns Logits (not predictions)

In [7]:
@torch.no_grad()
def get_logits_sp(model, processor, df, img_base, batch_size=4):
    """Single-pass digit logits. Returns {id: np.array(5)} with -inf padding."""
    logits_dict = {}
    for start in tqdm(range(0, len(df), batch_size), desc="SP", leave=False):
        batch_df = df.iloc[start:start + batch_size]
        images, texts, batch_ids, nc_list = [], [], [], []
        for _, row in batch_df.iterrows():
            images.append(Image.open(os.path.join(img_base, row['image_path'])).convert("RGB"))
            prompt = build_prompt(row)
            messages = [{"role": "user", "content": [
                {"type": "image"}, {"type": "text", "text": prompt}]}]
            texts.append(processor.apply_chat_template(messages, add_generation_prompt=True))
            batch_ids.append(row['id'])
            nc_list.append(row['num_choices'])

        inputs = processor(text=texts, images=images, return_tensors="pt",
                           padding=True, truncation=True, max_length=MAX_SEQ_LEN).to(device)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = model(**inputs)
        logits = outputs.logits

        for i in range(len(batch_df)):
            last_pos = inputs["attention_mask"][i].sum().item() - 1
            next_logits = logits[i, last_pos, :]
            nc = nc_list[i]
            scores = np.full(5, -1e9)
            for d in range(nc):
                scores[d] = next_logits[DIGIT_TOKEN_IDS[d]].float().item()
            logits_dict[batch_ids[i]] = scores
    return logits_dict

@torch.no_grad()
def get_logits_tta(model, processor, df, img_base, batch_size=4):
    """TTA digit logits (all N! permutations, mapped to original positions).
    Returns {id: np.array(5)} with -inf padding."""
    logits_dict = {}
    for idx in tqdm(range(len(df)), desc="TTA", leave=False):
        row = df.iloc[idx]
        nc = row['num_choices']
        choices = json.loads(row['choices'])
        all_perms = list(permutations(range(nc)))
        original_scores = np.zeros(5)
        original_scores[nc:] = -1e9
        image = Image.open(os.path.join(img_base, row['image_path'])).convert("RGB")

        for perm_start in range(0, len(all_perms), batch_size):
            perm_batch = all_perms[perm_start:perm_start + batch_size]
            images, texts, perms = [], [], []
            for perm in perm_batch:
                perm_row = row.copy()
                perm_row['choices'] = json.dumps([choices[perm[i]] for i in range(nc)])
                prompt = build_prompt(perm_row)
                messages = [{"role": "user", "content": [
                    {"type": "image"}, {"type": "text", "text": prompt}]}]
                texts.append(processor.apply_chat_template(messages, add_generation_prompt=True))
                images.append(image.copy())
                perms.append(perm)

            inputs = processor(text=texts, images=images, return_tensors="pt",
                               padding=True, truncation=True, max_length=MAX_SEQ_LEN).to(device)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(**inputs)
            logits = outputs.logits

            for i, perm in enumerate(perms):
                last_pos = inputs["attention_mask"][i].sum().item() - 1
                next_logits = logits[i, last_pos, :]
                for d in range(nc):
                    original_scores[perm[d]] += next_logits[DIGIT_TOKEN_IDS[d]].float().item()

        logits_dict[row['id']] = original_scores
    return logits_dict

def get_logits_hybrid(model, processor, df, img_base, batch_size=4):
    """SP for 2-choice, TTA for 3/4/5-choice. Returns {id: np.array(5)}."""
    df_2 = df[df['num_choices'] == 2].reset_index(drop=True)
    df_345 = df[df['num_choices'] >= 3].reset_index(drop=True)
    print(f"    2-choice (SP): {len(df_2)} | 3/4/5-choice (TTA): {len(df_345)}")

    logits = {}
    if len(df_2) > 0:
        logits.update(get_logits_sp(model, processor, df_2, img_base, batch_size))
    if len(df_345) > 0:
        logits.update(get_logits_tta(model, processor, df_345, img_base, batch_size))
    return logits

def accuracy_from_logits(logits_dict, df):
    """Compute accuracy from logits dict."""
    correct = 0
    for _, row in df.iterrows():
        nc = row['num_choices']
        pred = np.argmax(logits_dict[row['id']][:nc])
        if pred == int(row['answer']):
            correct += 1
    return correct / len(df)

def accuracy_by_nc(logits_dict, df):
    """Per num_choices accuracy."""
    results = {}
    for nc in sorted(df['num_choices'].unique()):
        sub = df[df['num_choices'] == nc]
        correct = sum(
            np.argmax(logits_dict[row['id']][:nc]) == int(row['answer'])
            for _, row in sub.iterrows()
        )
        results[nc] = (correct, len(sub), correct / len(sub))
    return results

print("Logit functions defined.")

Logit functions defined.


## Val: Run All Epochs + Ensemble

In [8]:
all_epoch_logits = {}

for epoch in ENSEMBLE_EPOCHS:
    print(f"\n{'='*50}")
    print(f"Epoch {epoch}")
    print(f"{'='*50}")
    model.set_adapter(f"epoch_{epoch}")
    model.eval()
    logits = get_logits_hybrid(model, processor, val_df, IMG_BASE, TTA_BATCH_SIZE)
    all_epoch_logits[epoch] = logits

    acc = accuracy_from_logits(logits, val_df)
    print(f"  Val Acc (hybrid TTA): {acc:.4f}")

print(f"\nAll {len(ENSEMBLE_EPOCHS)} epochs done.")


Epoch 3
    2-choice (SP): 244 | 3/4/5-choice (TTA): 804


SP:   0%|          | 0/61 [00:00<?, ?it/s]

TTA:   0%|          | 0/804 [00:00<?, ?it/s]

  Val Acc (hybrid TTA): 0.9046

Epoch 4
    2-choice (SP): 244 | 3/4/5-choice (TTA): 804


SP:   0%|          | 0/61 [00:00<?, ?it/s]

...

TTA:   0%|          | 0/804 [00:00<?, ?it/s]

  Val Acc (hybrid TTA): 0.9094

Epoch 6
    2-choice (SP): 244 | 3/4/5-choice (TTA): 804


SP:   0%|          | 0/61 [00:00<?, ?it/s]

TTA:   0%|          | 0/804 [00:00<?, ?it/s]

  Val Acc (hybrid TTA): 0.9084

All 4 epochs done.


In [9]:
from itertools import combinations

def ensemble_logits(epoch_logits_list, df):
    """Average logits from multiple epochs, return merged dict."""
    merged = {}
    for _, row in df.iterrows():
        qid = row['id']
        stacked = np.stack([el[qid] for el in epoch_logits_list])
        merged[qid] = stacked.mean(axis=0)
    return merged

print(f"{'Method':<25} {'Val Acc':>8}  {'2c':>7} {'3c':>7} {'4c':>7} {'5c':>7}")
print("=" * 75)

for epoch in ENSEMBLE_EPOCHS:
    acc = accuracy_from_logits(all_epoch_logits[epoch], val_df)
    by_nc = accuracy_by_nc(all_epoch_logits[epoch], val_df)
    nc_str = "  ".join(f"{by_nc[nc][2]:.4f}" for nc in [2, 3, 4, 5])
    print(f"E{epoch} (hybrid TTA)        {acc:>8.4f}  {nc_str}")

print("-" * 75)

best_acc, best_combo = 0, None

combos = []
for size in range(2, len(ENSEMBLE_EPOCHS) + 1):
    for combo in combinations(ENSEMBLE_EPOCHS, size):
        combos.append(list(combo))

for combo in combos:
    merged = ensemble_logits([all_epoch_logits[e] for e in combo], val_df)
    acc = accuracy_from_logits(merged, val_df)
    by_nc = accuracy_by_nc(merged, val_df)
    nc_str = "  ".join(f"{by_nc[nc][2]:.4f}" for nc in [2, 3, 4, 5])
    label = "+".join(f"E{e}" for e in combo)
    marker = ""
    if acc > best_acc:
        best_acc = acc
        best_combo = combo
        marker = " <-- best"
    print(f"{label:<25} {acc:>8.4f}  {nc_str}{marker}")

print("=" * 75)
print(f"\nBest ensemble: {'+'.join(f'E{e}' for e in best_combo)} = {best_acc:.4f}")
print(f"v19 single best (E5 SP):  0.9055")
print(f"v16-TTA Kaggle:           0.92354")
print(f"v19-hybridTTA Kaggle:     0.92354")

Method                     Val Acc       2c      3c      4c      5c
E3 (hybrid TTA)          0.9046  0.8975  0.9370  0.8929  0.6364
E4 (hybrid TTA)          0.9074  0.9016  0.9390  0.9008  0.6136
E5 (hybrid TTA)          0.9094  0.9057  0.9429  0.8968  0.6136
E6 (hybrid TTA)          0.9084  0.9016  0.9429  0.8968  0.6136
---------------------------------------------------------------------------
E3+E4                       0.9103  0.9057  0.9390  0.9048  0.6364 <-- best
E3+E5                       0.9122  0.9180  0.9409  0.9008  0.6136 <-- best
E3+E6                       0.9122  0.9180  0.9409  0.9008  0.6136
E4+E5                       0.9084  0.9016  0.9409  0.8968  0.6364
E4+E6                       0.9094  0.9016  0.9429  0.8968  0.6364
E5+E6                       0.9084  0.9016  0.9429  0.8968  0.6136
E3+E4+E5                    0.9094  0.9057  0.9409  0.9008  0.6136
E3+E4+E6                    0.9094  0.9057  0.9390  0.9008  0.6364
E3+E5+E6                    0.9113  0.9098  0.

## Test: Ensemble Predictions

Uses the best combination from validation. Change SUBMIT_EPOCHS below for a different combination.

In [10]:
SUBMIT_EPOCHS = best_combo
print(f"Submitting ensemble: {'+'.join(f'E{e}' for e in SUBMIT_EPOCHS)}")

test_epoch_logits = {}
for epoch in SUBMIT_EPOCHS:
    print(f"\n{'='*50}")
    print(f"Test — Epoch {epoch}")
    print(f"{'='*50}")
    model.set_adapter(f"epoch_{epoch}")
    model.eval()
    logits = get_logits_hybrid(model, processor, test_df, IMG_BASE, TTA_BATCH_SIZE)
    test_epoch_logits[epoch] = logits
    print(f"  Done. {len(logits)} predictions.")

test_merged = ensemble_logits([test_epoch_logits[e] for e in SUBMIT_EPOCHS], test_df)

test_preds = {}
for _, row in test_df.iterrows():
    nc = row['num_choices']
    test_preds[row['id']] = int(np.argmax(test_merged[row['id']][:nc]))

pred_dist = pd.Series(list(test_preds.values())).value_counts().sort_index()
print(f"\nPrediction distribution: {dict(pred_dist)}")

Submitting ensemble: E3+E4+E5+E6

Test — Epoch 3
    2-choice (SP): 272 | 3/4/5-choice (TTA): 736


SP:   0%|          | 0/68 [00:00<?, ?it/s]

TTA:   0%|          | 0/736 [00:00<?, ?it/s]

  Done. 1008 predictions.

Test — Epoch 4
    2-choice (SP): 272 | 3/4/5-choice (TTA): 736


SP:   0%|          | 0/68 [00:00<?, ?it/s]

...

TTA:   0%|          | 0/736 [00:00<?, ?it/s]

  Done. 1008 predictions.

Test — Epoch 6
    2-choice (SP): 272 | 3/4/5-choice (TTA): 736


SP:   0%|          | 0/68 [00:00<?, ?it/s]

TTA:   0%|          | 0/736 [00:00<?, ?it/s]

  Done. 1008 predictions.

Prediction distribution: {0: np.int64(356), 1: np.int64(353), 2: np.int64(221), 3: np.int64(75), 4: np.int64(3)}


In [11]:
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
submission = pd.DataFrame({
    "id": [row['id'] for _, row in test_df.iterrows()],
    "answer": [test_preds[row['id']] for _, row in test_df.iterrows()]
})
assert set(submission['id']) == set(sample_sub['id']), "ID mismatch!"
submission = submission.set_index('id').loc[sample_sub['id']].reset_index()

combo_label = "_".join(f"e{e}" for e in SUBMIT_EPOCHS)
fname = f"submission_ensemble_{combo_label}.csv"
submission.to_csv(fname, index=False)
print(f"Saved {fname}")
print(submission.head(10))

Saved submission_ensemble_e3_e4_e5_e6.csv
           id  answer
0  test_01750       2
1  test_00128       0
2  test_02891       3
3  test_02425       1
4  test_00930       2
5  test_03725       2
6  test_00009       1
7  test_02880       0
8  test_01208       0
9  test_00619       1


In [12]:
from google.colab import files
files.download(fname)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>